# Analisi domini SNI 

Per bloccare un'app specifica con lo stesso
meccanismo già usato da OAF, serve una regola basata su dominio, come quella già
esistente per AppStore/GooglePlay.  

## Obiettivo
> **Quanti domini diversi usa davvero l'app durante una sessione?** 

Un dominio dominante rende la regola affidabile; tanti domini sparsi la rendono
inaffidabile.

Nel traffico cifrato, il primo pacchetto di ogni connessione
HTTPS (il ClientHello TLS) contiene il nome del dominio richiesto in
chiaro, nell'estensione SNI (Server Name Indication).

## Cosa guardare nel risultato
- **Un dominio nettamente dominante** (es. >70-80% delle connessioni): buon
  candidato per una regola OAF dedicata a quell'app
- **Pochi domini stabili** (2-3, sempre gli stessi tra le ripetizioni): va
  bene comunque, puoi aggiungere più regole (più appid) alla stessa app_list,
  esattamente come oggi `oaf_block.sh` mette insieme 7002 e 9001 per bloccare
  sia AppStore che GooglePlay in un colpo solo
- **Tanti domini diversi, che cambiano da una ripetizione all'altra** (tipico
  di app che appoggiano su CDN generici tipo Cloudflare/Akamai/AWS condivisi
  con altri servizi): la regola rischia di essere inaffidabile o di bloccare
  anche altro, in questo caso ha più senso restare sul blocco store-level.

## 1. Setup

In [2]:
import struct
import glob
from collections import Counter

risultati_raggruppati = {}

## 2. Estrazione SNI da una cattura

Viene letta una cattura `.pcap` pacchetto per pacchetto (header letti con `struct`, non decodifica
completa) e, per ogni pacchetto che è l'inizio di un handshake TLS
(ClientHello), estrae il dominio richiesto dall'estensione SNI.

In [3]:
def estrai_sni_da_payload(payload):
    if len(payload) < 6:
        return None
    if payload[0] != 0x16:       
        return None
    if payload[5] != 0x01:       
        return None
    try:
        pos = 5
        pos += 4                 
        pos += 2                
        pos += 32                
        session_id_len = payload[pos]
        pos += 1 + session_id_len
        cipher_len = int.from_bytes(payload[pos:pos+2], "big")
        pos += 2 + cipher_len
        compression_len = payload[pos]
        pos += 1 + compression_len
        if pos + 2 > len(payload):
            return None
        ext_total_len = int.from_bytes(payload[pos:pos+2], "big")
        pos += 2
        end = pos + ext_total_len
        while pos + 4 <= len(payload) and pos < end:
            ext_type = int.from_bytes(payload[pos:pos+2], "big")
            ext_len = int.from_bytes(payload[pos+2:pos+4], "big")
            ext_data = payload[pos+4:pos+4+ext_len]
            if ext_type == 0x0000 and len(ext_data) >= 5:  
                name_len = int.from_bytes(ext_data[3:5], "big")
                hostname = ext_data[5:5+name_len]
                return hostname.decode("ascii", errors="ignore")
            pos += 4 + ext_len
    except Exception:
        return None
    return None


def estrai_domini_da_cattura(percorso_file):
    domini = []
    with open(percorso_file, "rb") as f:
        header_globale = f.read(24)
        if len(header_globale) < 24:
            return domini
        magic = struct.unpack("I", header_globale[:4])[0]
        endian = "<" if magic == 0xa1b2c3d4 else ">"

        while True:
            header_pacchetto = f.read(16)
            if len(header_pacchetto) < 16:
                break
            _, _, len_catturata, _ = struct.unpack(endian + "IIII", header_pacchetto)
            dati = f.read(len_catturata)

            if len(dati) < 34:
                continue
            ip_header = dati[14:34]
            if ip_header[0] >> 4 != 4:      
                continue
            ihl = (ip_header[0] & 0x0F) * 4
            proto = ip_header[9]
            if proto != 6:                  
                continue
            tcp_start = 14 + ihl
            if len(dati) < tcp_start + 20:
                continue
            data_offset = (dati[tcp_start + 12] >> 4) * 4
            payload_start = tcp_start + data_offset
            payload = dati[payload_start:]

            sni = estrai_sni_da_payload(payload)
            if sni:
                domini.append(sni)

    return domini

## 3. Analisi su più catture della stessa app

Si passa una lista di percorsi (tutte le ripetizioni di una stessa app) e si analizza
la distribuzione dei domini richiesti, in ordine di frequenza.

In [4]:
def analizza_app(percorsi_pcap, nome_app="app"):
    contatore = Counter()
    for percorso in percorsi_pcap:
        domini = estrai_domini_da_cattura(percorso)
        contatore.update(domini)

    totale = sum(contatore.values())
    print(f"=== {nome_app}: {totale} ClientHello trovati in {len(percorsi_pcap)} catture ===\n")
    if totale == 0:
        print("Nessun ClientHello TLS trovato")
        return contatore

    for dominio, conteggio in contatore.most_common(15):
        perc = 100 * conteggio / totale
        barra = "#" * int(perc / 2)
        print(f"{perc:5.1f}%  {barra:<50} {dominio}  ({conteggio})")

    dominio_top, conteggio_top = contatore.most_common(1)[0]
    perc_top = 100 * conteggio_top / totale
    print(f"\nDominio piu' frequente: {dominio_top} ({perc_top:.1f}% delle connessioni)")

    return contatore

def dominio_base(hostname):
    parti = hostname.split(".")
    if len(parti) < 2:
        return hostname
    return ".".join(parti[-2:])


def analizza_app_raggruppato(percorsi_pcap, nome_app="app"):
    contatore = Counter()
    for percorso in percorsi_pcap:
        domini = estrai_domini_da_cattura(percorso)
        contatore.update(dominio_base(d) for d in domini)

    totale = sum(contatore.values())
    print(f"=== {nome_app} (raggruppato per dominio radice): {totale} ClientHello in {len(percorsi_pcap)} catture ===\n")
    if totale == 0:
        print("Nessun ClientHello TLS trovato.")
        return contatore

    for dominio, conteggio in contatore.most_common(15):
        perc = 100 * conteggio / totale
        barra = "#" * int(perc / 2)
        print(f"{perc:5.1f}%  {barra:<50} {dominio}  ({conteggio})")

    return contatore

## 4. Criteri di selezione dei casi studio

La scelta di focalizzare l'analisi dei domini SNI su IKEA e Unieuro è stata guidata da una valutazione preliminare sulla rilevanza statistica del traffico generato dalle applicazioni disponibili. Si è operata una selezione basata sui seguenti criteri metodologici:

*   **Rilevanza del volume di traffico**: Applicazioni come *Calcolatrice* o *Shazam* sono state escluse in quanto generano un volume di dati troppo esiguo per consentire un'analisi significativa delle connessioni TLS.
*   **Complessità e dipendenza dai servizi**: Applicazioni come *Corriere della Sera* sono caratterizzate da una frammentazione eccessiva del traffico, dovuta alla forte dipendenza da domini terzi (CDN, pubblicità, contenuti multimediali), che rende difficile isolare il dominio proprietario. *Disney+*, invece, è stata esclusa per la necessità di autenticazione immediata, che limita l'osservazione del traffico di navigazione standard.
*   **Idoneità per l'enforcement**: IKEA e Unieuro rappresentano i casi più promettenti: presentano volumi di traffico sufficienti e strutture di connessione più coerenti, rendendole le candidate ideali per validare l'efficacia di regole di blocco mirate (per app) rispetto al semplice blocco di ripiego a livello di store.

### Risultati Parziali

In [5]:
percorsi = sorted(glob.glob("catture/*ikea*.pcap"))
print(f"Trovati {len(percorsi)} file:")
for p in percorsi:
    print(f"  {p}")

if percorsi:
    contatore = analizza_app(percorsi, nome_app="IKEA")

Trovati 8 file:
  catture\cattura_uso_ikea1.pcap
  catture\cattura_uso_ikea2.pcap
  catture\cattura_uso_ikea3.pcap
  catture\cattura_uso_ikea4.pcap
  catture\cattura_uso_ikea5.pcap
  catture\cattura_uso_ikea6.pcap
  catture\cattura_uso_ikea7.pcap
  catture\cattura_uso_ikea8.pcap
=== IKEA: 341 ClientHello trovati in 8 catture ===

 52.2%  ##########################                         logx.optimizely.com  (178)
 11.4%  #####                                              shop.static.ingka.ikea.com  (39)
  4.7%  ##                                                 cdn.optimizely.com  (16)
  4.7%  ##                                                 shop.api.ingka.ikea.com  (16)
  2.9%  #                                                  www.ikea.com  (10)
  2.6%  #                                                  web-api.ikea.com  (9)
  2.6%  #                                                  att-104451-prod.approovr.io  (9)
  2.3%  #                                                  mobile-

In [6]:
percorsi = sorted(glob.glob("catture/*unieuro*.pcap"))
print(f"Trovati {len(percorsi)} file:")
for p in percorsi:
    print(f"  {p}")

if percorsi:
    contatore = analizza_app(percorsi, nome_app="UNIEURO")

Trovati 8 file:
  catture\cattura_uso_unieuro1.pcap
  catture\cattura_uso_unieuro2.pcap
  catture\cattura_uso_unieuro3.pcap
  catture\cattura_uso_unieuro4.pcap
  catture\cattura_uso_unieuro5.pcap
  catture\cattura_uso_unieuro6.pcap
  catture\cattura_uso_unieuro7.pcap
  catture\cattura_uso_unieuro8.pcap
=== UNIEURO: 285 ClientHello trovati in 8 catture ===

 24.6%  ############                                       analytics.eu.adjust.com  (70)
  5.6%  ##                                                 gcp-eu-cdn.contentstack.com  (16)
  5.6%  ##                                                 mnbcenyfii-dsn.algolia.net  (16)
  5.6%  ##                                                 www.paypal.com  (16)
  5.6%  ##                                                 www.unieuro.it  (16)
  5.3%  ##                                                 api.radar.io  (15)
  4.2%  ##                                                 static1.unieuro.it  (12)
  3.5%  #                                    

## 5. Analisi raggruppata per dominio radice

Il dominio "più frequente" spesso appartiene a un SDK di terze parti
(analytics, A/B testing, attribution) che l'app integra, non all'azienda
proprietaria dell'app, quindi non è detto sia il candidato giusto per un
blocco che voglia davvero fermare il funzionamento dell'app.

Questa vista raggruppa i sottodomini sotto lo stesso dominio radice
(es. `shop.api.ingka.ikea.com` → `ikea.com`), per separare meglio "traffico
verso l'azienda" da "traffico verso servizi terzi integrati".

In [7]:
percorsi = sorted(glob.glob("catture/*ikea*.pcap"))
if percorsi:
    risultati_raggruppati["IKEA"] = analizza_app_raggruppato(percorsi, nome_app="IKEA")

=== IKEA (raggruppato per dominio radice): 341 ClientHello in 8 catture ===

 56.9%  ############################                       optimizely.com  (194)
 27.3%  #############                                      ikea.com  (93)
  5.0%  ##                                                 approovr.io  (17)
  2.3%  #                                                  content-square.net  (8)
  2.3%  #                                                  microsoft.com  (8)
  1.2%                                                     googleapis.com  (4)
  0.9%                                                     apple.com  (3)
  0.9%                                                     spotify.com  (3)
  0.3%                                                     airbnb.it  (1)
  0.3%                                                     gstatic.com  (1)
  0.3%                                                     muscache.com  (1)
  0.3%                                                     cookielaw.org  

In [8]:
percorsi = sorted(glob.glob("catture/*unieuro*.pcap"))
if percorsi:
    risultati_raggruppati["Unieuro"] = analizza_app_raggruppato(percorsi, nome_app="Unieuro")

=== Unieuro (raggruppato per dominio radice): 285 ClientHello in 8 catture ===

 24.6%  ############                                       adjust.com  (70)
 13.7%  ######                                             unieuro.it  (39)
  9.5%  ####                                               contentstack.com  (27)
  5.6%  ##                                                 algolia.net  (16)
  5.6%  ##                                                 paypal.com  (16)
  5.3%  ##                                                 radar.io  (15)
  3.5%  #                                                  onetrust.io  (10)
  3.2%  #                                                  swogo.net  (9)
  3.2%  #                                                  googleapis.com  (9)
  2.8%  #                                                  nip.io  (8)
  2.8%  #                                                  unieuro.com  (8)
  2.8%  #                                                  bstatic.com  (8)
  2.5%

## 6. Considerazioni conclusive (parziali)

**Criterio usato per giudicare un'app "buona candidata" al blocco per-app**: il traffico
verso il dominio radice dell'azienda proprietaria (non SDK di terze parti come
analytics) deve essere presente in modo consistente e non trascurabile, una
soglia indicativa attorno al 20-25% è un buon punto di partenza per considerare un dominio
"abbastanza dominante" da giustificare una regola di blocco dedicata, ma va sempre
verificato empiricamente sul testbed (come fatto per IKEA) prima di trarre conclusioni
definitive, perché il conteggio di ClientHello non misura quanto quel traffico sia
funzionalmente critico per l'app.

In [9]:
DOMINIO_PROPRIO = {
    "IKEA": ["ikea.com"],
    "Unieuro": ["unieuro.it", "unieuro.com"],
}


def valuta_candidatura(nome_app, soglia=20):
    if nome_app not in risultati_raggruppati:
        return None
    contatore = risultati_raggruppati[nome_app]
    totale = sum(contatore.values())
    if totale == 0:
        return {"app": nome_app, "percentuale": 0.0, "verdetto": "nessun dato"}

    domini_propri = DOMINIO_PROPRIO.get(nome_app, [])
    conteggio_proprio = sum(contatore.get(d, 0) for d in domini_propri)
    perc = 100 * conteggio_proprio / totale

    verdetto = "buona candidata" if perc >= soglia else "candidata debole (traffico troppo disperso)"
    return {"app": nome_app, "percentuale": perc, "verdetto": verdetto}


print(f"{'App':<12}{'Dominio/i proprio/i':<28}{'% traffico proprio':<20}Verdetto")
print("-" * 80)
for nome_app in risultati_raggruppati:
    r = valuta_candidatura(nome_app)
    domini_str = ", ".join(DOMINIO_PROPRIO.get(nome_app, ["?"]))
    print(f"{r['app']:<12}{domini_str:<28}{r['percentuale']:>6.1f}%{'':<13}{r['verdetto']}")

App         Dominio/i proprio/i         % traffico proprio  Verdetto
--------------------------------------------------------------------------------
IKEA        ikea.com                      27.3%             buona candidata
Unieuro     unieuro.it, unieuro.com       16.5%             candidata debole (traffico troppo disperso)
